# Flux post-processing chain: IRGA72 NEE (2016-2026.09)

**notebook version**: `1` (25 Oct 2026), adapted from the diive notebook `FluxPostProcessingChain.ipynb`

## ℹ️ About this notebook

Post-processes the Level-1 CO2 flux (`FC`) of the **IRGA72** closed-path analyser into net ecosystem
exchange (`NEE`), gap-fills it and partitions it into GPP and RECO. The levels are those of the
Swiss FluxNet workflow:

| Level | What it does | Function |
|---|---|---|
| L2 | EddyPro quality-flag expansion and QCF | `run_level2` |
| L3.1 | Storage correction (`FC` -> `NEE`) | `run_level31` |
| L3.2 | Outlier removal | `make_level32_detector` + `run_level32` |
| L3.3 | USTAR filtering, three constant thresholds | `run_level33_constant_ustar` |
| L4.1 | Gap-filling with XGBoost | `run_level41_xgb` |
| L4.2 | Partitioning into GPP and RECO, nighttime and daytime (ONEFlux) | `run_level42_partitioning` |

Each level is a plain function that takes the `FluxLevelData` container and returns a **new**
one. Nothing is mutated in place, so a level can be re-run without repeating the levels above it.

The settings follow the 2016-2024 notebooks of this stage (`01_FPC_L3.3_…` and `03_FPC_L4.1_…`),
carried to the diive API of v0.91.0 and later.

***

## 📇 Data sources

`20_MERGE_DATA/IRGA72/22.3_FLUXES_L1_IRGA72_and_METEO_2016-2026.09.parquet` in the external data
folder: the merged Level-1 fluxes of all 18 `IRGA72` runs, joined with the merged meteo products of
`10_METEO/30_PRODUCTS/99`. It runs from 1 January 2016 00:15 to 30 September 2026 23:45
(`TIMESTAMP_MIDDLE`, UTC+01:00); the flux columns are empty until the first `IRGA72` record on
11 January 2016.

**Meteo drivers.** Gap-filling and partitioning use the homogenised air temperature
(`TA_T1_47_1_HOMOGENIZED_gfXG`) and vapour pressure deficit (`VPD_T1_47_1_HOMOGENIZED`), which put
the probe eras of 2016 and June 2026 on one level, and the gap-filled incoming shortwave radiation
(`SW_IN_T1_47_1_gfXG`). All three are complete. The ONEFlux partitioning also needs *measured*
temperature and radiation; these are the same columns restricted to the records their
`ISFILLED` flag marks as measured at 47 m (code `0`), built below and added with `add_driver`.

**Periods to keep in mind.**

- The commissioning of the LI-7200 (11 January to 25 May 2016) is removed by hand in Level 3.2,
  as in the 2016-2024 notebooks: water in the filter suppressed the fluxes.
- 19 May to 7 June 2018 has no `FC`: the flow module had failed and the run of that period is
  sonic-only.
- All `IRGA72` CO2 concentrations between 14 December 2017 and 15 March 2019 were measured
  against a wrong calibration gas (factor 0.974 on the concentration, to be applied before the flux
  calculation). Whether the Level-1 runs applied it is not established, so `FC` in that period is
  not corrected here.

## ⚙️ Settings

In [ ]:
# --- Source ------------------------------------------------------------------------------------
DATADIR = (r"F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data"
           r"\workflow")
SOURCEFILE = rf"{DATADIR}\20_MERGE_DATA\IRGA72\22.3_FLUXES_L1_IRGA72_and_METEO_2016-2026.09.parquet"

# --- Output ------------------------------------------------------------------------------------
OUTNAME = "01_FPPC_IRGA72_2016-2026.09"  # same name as this notebook
OUTPATH = rf"{DATADIR}\30_FLUX_PROCESSING_CHAIN\34_IRGA72_2016-2024"

# --- Site and flux -----------------------------------------------------------------------------
FLUXCOL = 'FC'  # CO2 flux of the closed-path LI-7200; no self-heating correction is needed
SITE_LAT = 47.478333  # CH-LAE
SITE_LON = 8.364389
UTC_OFFSET = 1  # timestamps are UTC+01:00 all year
NIGHTTIME_THRESHOLD = 20  # W m-2 potential radiation below this = nighttime
DAYTIME_ACCEPT_QCF_BELOW = 2  # keep QCF 0 and 1
NIGHTTIME_ACCEPT_QCF_BELOW = 2

# --- Level 2 -----------------------------------------------------------------------------------
LEVEL2_SETTINGS = {
    'signal_strength': {'apply': True,
                        'signal_strength_col': 'CUSTOM_SIGNAL_STRENGTH_IRGA72_MEAN',
                        'method': 'discard below',
                        'threshold': 60},
    'raw_data_screening_vm97': {'apply': True,
                                'spikes': True, 'amplitude': False, 'dropout': True,
                                'abslim': False, 'skewkurt_hf': False, 'skewkurt_sf': False,
                                'discont_hf': False, 'discont_sf': False},
    'ssitc': {'apply': True, 'setflag_timeperiod': None},
    'gas_completeness': {'apply': False},
    'spectral_correction_factor': {'apply': True},
    'angle_of_attack': {'apply': False, 'application_dates': []},
    'steadiness_of_horizontal_wind': {'apply': False},
}

# --- Level 3.2 ---------------------------------------------------------------------------------
# The LI-7200 commissioning of 2016 (inclusive date ranges).
REMOVE_DATES = [
    ['2016-01-11', '2016-01-22'],
    ['2016-03-01', '2016-05-25'],
]

# --- Level 3.3 ---------------------------------------------------------------------------------
# Constant USTAR thresholds (m s-1), the 16th, 50th and 84th percentiles of the bootstrap
# detection of the 2016-2024 record (stage 32_). They have not been re-derived on the re-run
# Level-1 record of 2016-2026.09.
USTAR_THRESHOLDS = [0.271619922, 0.303628125, 0.339684084]
USTAR_SCENARIOS = ['CUT_16', 'CUT_50', 'CUT_84']

# --- Level 4.1 and 4.2: meteo drivers ----------------------------------------------------------
TA_F = 'TA_T1_47_1_HOMOGENIZED_gfXG'    # degC, complete
SW_IN_F = 'SW_IN_T1_47_1_gfXG'          # W m-2, complete
VPD_F = 'VPD_T1_47_1_HOMOGENIZED'       # kPa, complete
TA_FLAG = 'FLAG_TA_T1_47_1_ISFILLED'    # 0 = measured at 47 m
SW_IN_FLAG = 'FLAG_SW_IN_T1_47_1_ISFILLED'
TA_MEAS = 'TA_T1_47_1_HOMOGENIZED_MEASURED'  # built below from TA_F and TA_FLAG
SW_IN_MEAS = 'SW_IN_T1_47_1_MEASURED'        # built below from SW_IN_F and SW_IN_FLAG
FEATURES = [TA_F, SW_IN_F, VPD_F]

# XGBoost hyperparameters, at the production values suggested by the diive notebook.
XGB_SETTINGS = dict(n_estimators=350, max_depth=6, learning_rate=0.05,
                    early_stopping_rounds=10, random_state=42, n_jobs=-1, shap_max_rows=5000)

In [ ]:
from datetime import datetime
import importlib.metadata

import diive as dv
from diive.core.ml.feature_engineer import FeatureEngineer
from diive.flux.postprocessingchain import (
    add_driver,
    init_flux_data,
    make_level32_detector,
    run_level2,
    run_level31,
    run_level32,
    run_level33_constant_ustar,
    run_level41_xgb,
    run_level42_partitioning,
    # Reporting over the finished chain
    gapfilled_variables,
    plot_feature_ranks_per_year,
    plot_gapfilled_cumulative,
    report_gapfilling_feature_importances,
    report_gapfilling_model_scores,
    report_gapfilling_poolyears,
    report_gapfilling_variables,
    report_traintest_details,
    report_traintest_model_scores,
)

# Start time is kept so the last cell can report the total runtime of the notebook.
NOTEBOOK_START = datetime.now()
print(f"Last run: {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive version: v{importlib.metadata.version('diive')}")

## 📂 Load data

`init_flux_data` computes `SW_IN_POT`, `DAYTIME` and `NIGHTTIME` itself and raises if any of them
already exists, so they are dropped first if the input carries them.

In [ ]:
df = dv.load_parquet(filepath=SOURCEFILE)
df = df.drop(columns=[c for c in ('SW_IN_POT', 'DAYTIME', 'NIGHTTIME') if c in df.columns])

for _c in [FLUXCOL, TA_F, SW_IN_F, VPD_F, TA_FLAG, SW_IN_FLAG,
           LEVEL2_SETTINGS['signal_strength']['signal_strength_col']]:
    assert _c in df.columns, f"{_c} is not in {SOURCEFILE}"
for _c in FEATURES:
    assert df[_c].notna().all(), f"driver {_c} has gaps"
print(f"{len(df):,} records x {df.shape[1]} columns, {df.index[0]} -> {df.index[-1]}")
print(f"{FLUXCOL}: {df[FLUXCOL].notna().sum():,} records")

## Step 1 - initialise the container

Calculates potential radiation, derives the day/night flags and freezes the site metadata that every
later level reads. `*_ACCEPT_QCF_BELOW = 2` keeps QCF 0 (all tests pass) and QCF 1 (soft warnings).

In [ ]:
data = init_flux_data(
    df=df,
    fluxcol=FLUXCOL,
    site_lat=SITE_LAT,
    site_lon=SITE_LON,
    utc_offset=UTC_OFFSET,
    nighttime_threshold=NIGHTTIME_THRESHOLD,
    daytime_accept_qcf_below=DAYTIME_ACCEPT_QCF_BELOW,
    nighttime_accept_qcf_below=NIGHTTIME_ACCEPT_QCF_BELOW,
)
print(data)

## Step 2 - Level 2: quality flag expansion

The tests of the 2016-2024 notebooks: steady state and integral turbulence characteristics (SSITC),
the spectral correction factor, the LI-7200 signal strength (records below 60 discarded), and the
VM97 raw-data screening for spikes and dropouts. Gas completeness is not tested.

In [ ]:
data = run_level2(data, **LEVEL2_SETTINGS)
print(f'After L2: {data.filteredseries.dropna().count():,} valid records')

### 📊 Plot: Level-2 QCF heatmaps

In [ ]:
data.levels.level2_qcf.showplot_qcf_heatmaps()

### 📋 Table: Level-2 flags

In [ ]:
data.levels.level2_qcf.report_qcf_flags()

## Step 3 - Level 3.1: storage correction

Adds the single-point storage term to the flux (`NEE = FC + SC_SINGLE`), with gaps in the storage
term filled.

In [ ]:
data = run_level31(data, gapfill_storage_term=True, set_storage_to_zero=False)
print(f'After L3.1: {data.filteredseries.dropna().count():,} valid records '
      f'(col: {data.filteredseries.name})')

## Step 4 - Level 3.2: outlier removal

The sequence of the 2016-2024 notebook: the manual removal of the 2016 commissioning, two Hampel
passes (the first on differenced data, 4 sigma day and night; the second on the data, 5.5 sigma by
day and 3.5 by night), a lower limit of -10 at night, absolute limits of ±50, and a local
standard-deviation test at 5.5 SD. All windows are 13 days. Each test is followed by `addflag()`.

In [ ]:
data, sod = make_level32_detector(data)

sod.flag_manualremoval_test(remove_dates=REMOVE_DATES, showplot=False, verbose=True)
sod.addflag()

sod.flag_outliers_hampel_test(window_length=48 * 13, n_sigma_daytime=4, n_sigma_nighttime=4,
                              use_differencing=True, separate_day_night=True,
                              showplot=False, verbose=True, repeat=True)
sod.addflag()

sod.flag_outliers_hampel_test(window_length=48 * 13, n_sigma_daytime=5.5, n_sigma_nighttime=3.5,
                              use_differencing=False, separate_day_night=True,
                              showplot=False, verbose=True, repeat=True)
sod.addflag()

sod.flag_outliers_trim_low_test(trim_nighttime=True, lower_limit=-10, showplot=False, verbose=True)
sod.addflag()

sod.flag_outliers_abslim_test(minval=-50, maxval=50, showplot=False, verbose=True)
sod.addflag()

sod.flag_outliers_localsd_test(n_sd_daytime=5.5, n_sd_nighttime=5.5, winsize=48 * 13,
                               constant_sd=False, separate_day_night=True,
                               showplot=False, verbose=True, repeat=True)
sod.addflag()

data = run_level32(data, outlier_detector=sod)
print(f'After L3.2: {data.filteredseries.dropna().count():,} valid records')

### 📊 Plot: Level-3.2 QCF heatmaps

In [ ]:
data.levels.level32_qcf.showplot_qcf_heatmaps()

## Step 5 - Level 3.3: USTAR filtering

Three scenarios, one per constant threshold: the 16th, 50th and 84th percentiles of the USTAR
detection. After this call `data.filteredseries` is `None`, because there is no single filtered
series across scenarios; the series are read per scenario.

In [ ]:
data = run_level33_constant_ustar(
    data,
    thresholds=USTAR_THRESHOLDS,
    threshold_labels=USTAR_SCENARIOS,
    showplot=False,
    verbose=True,
)

for _scen in USTAR_SCENARIOS:
    _flux = data.levels.filteredseries_level33_qcf[_scen]
    _flux_hq = data.levels.filteredseries_level33_hq[_scen]  # QCF=0 only
    print(f'After L3.3 ({_scen}): {_flux.dropna().count():,} accepted | '
          f'{_flux_hq.dropna().count():,} high-quality')

### 📋 Table: gaps before gap-filling

`data.gap_stats(level)` returns one `GapStats` per scenario.

In [ ]:
for _scen, _gs in data.gap_stats('L3.3').items():
    _gs.report()

## Step 6 - drivers and feature engineering

The measured temperature and radiation for the ONEFlux partitioning are the complete driver columns
restricted to the records measured at 47 m. They go into `data.full_df` with `add_driver`, because
L4.1 and L4.2 read their drivers from there.

The feature engineer is built once and handed to the XGBoost gap-filler. `target_col` is a
placeholder; L4.1 applies the engineer to the predictors only.

In [ ]:
_ta_meas = df[TA_F].where(df[TA_FLAG] == 0).rename(TA_MEAS)
_sw_meas = df[SW_IN_F].where(df[SW_IN_FLAG] == 0).rename(SW_IN_MEAS)
data = add_driver(data, _ta_meas)
data = add_driver(data, _sw_meas)
print(f"{TA_MEAS}: {_ta_meas.notna().sum():,} measured records of {len(_ta_meas):,}")
print(f"{SW_IN_MEAS}: {_sw_meas.notna().sum():,} measured records of {len(_sw_meas):,}")

engineer = FeatureEngineer(
    target_col='_target_',
    features_lag=[-2, 2],
    features_lag_stepsize=1,
    features_rolling=[2, 4, 12, 24, 48],
    features_rolling_stats=['median', 'min', 'max', 'std'],
    features_ema=[6, 12, 24, 48],
    vectorize_timestamps=True,
    add_continuous_record_number=True,
    sanitize_timestamp=True,
    verbose=1,
)

## Step 7 - Level 4.1: gap-filling with XGBoost

One long-term model per USTAR scenario, with SHAP-based feature reduction.

In [ ]:
data = run_level41_xgb(
    data,
    features=FEATURES,
    engineer=engineer,
    reduce_features=True,
    verbose=1,
    **XGB_SETTINGS,
)

## Step 8 - Level 4.2: NEE partitioning

Partitions NEE into GPP and RECO, once per USTAR scenario, with both methods in their ONEFlux
implementation (as used by ICOS and FLUXNET): nighttime (Reichstein et al. 2005, temperature
response of nighttime NEE) and daytime (Lasslop et al. 2010, light-response curve of daytime NEE).
Both read the XGBoost gap-filled NEE of Level 4.1. The output columns carry the variant as a suffix
(`_NT_OF`, `_DT_OF`).

In [ ]:
data = run_level42_partitioning(
    data,
    method='nighttime',
    mode='oneflux',
    ta=TA_MEAS,
    sw_in=SW_IN_MEAS,
    ta_f=TA_F,
    gapfill_method='xgb',
)

In [ ]:
data = run_level42_partitioning(
    data,
    method='daytime',
    mode='oneflux',
    ta=TA_MEAS,
    sw_in=SW_IN_MEAS,
    ta_f=TA_F,
    sw_in_f=SW_IN_F,
    vpd_f=VPD_F,  # kPa
    gapfill_method='xgb',
)
data.partitioned_cols()

## Step 9 - what came out

`gapfilled_cols()` gives the output column per method and scenario; `nongapfilled_cols()` gives the
measured column each one filled.

In [ ]:
print(data.gapfilled_cols())
print(data.nongapfilled_cols())

### 📋 Table: gap-filled variables

In [ ]:
report_gapfilling_variables(data)

### 📋 Table: statistics of the gap-filled variables

In [ ]:
gapfilled_variables(data).describe()

## Step 10 - model reporting

Held-out test scores first. These come from a random train/test split of the complete rows, which
is the split that reproduces the gap-filling task.

### 📋 Table: held-out test scores

In [ ]:
report_traintest_model_scores(data)

### 📋 Table: train/test details

In [ ]:
report_traintest_details(data)

In-sample scores are optimistically biased; compare them with the held-out scores above.

### 📋 Table: in-sample gap-filling scores

In [ ]:
report_gapfilling_model_scores(data)

### 📋 Table: years pooled for each year's model

In [ ]:
report_gapfilling_poolyears(data)

### 📋 Table: SHAP feature importances per year

In [ ]:
report_gapfilling_feature_importances(data)

### 📊 Plot: feature ranks per year

In [ ]:
plot_feature_ranks_per_year(data)

## Step 11 - result plots

### 📊 Plot: measured and gap-filled NEE, CUT_50

In [ ]:
data.plot_gapfilled_heatmaps(ustar_scenario='CUT_50')

### 📊 Plot: cumulative NEE, CUT_50

In [ ]:
UMOL_TO_GC = 12.011 * 1e-6 * 1800  # umol CO2 m-2 s-1 over 30 min -> gC m-2

data.plot_cumulative_comparison(
    ustar_scenario='CUT_50',
    conv_factor=UMOL_TO_GC,
    units='gC m-2',
)

### 📊 Plot: cumulative of each gap-filled variable per year

Each year is drawn as its own line, with a multi-year reference band. 2016 starts on 11 January and
2026 ends on 30 September, so their totals are not comparable with the full years.

In [ ]:
plot_gapfilled_cumulative(
    data,
    gain=UMOL_TO_GC,
    units='gC m-2',
    per_year=True,
)

## 💾 Step 12 - export

`merged_df()` puts the untouched input columns and everything the chain produced into one
dataframe. Nothing in the input is overwritten; the chain only adds columns.

In [ ]:
out_df = data.merged_df()
print(f"{out_df.shape[0]:,} records x {out_df.shape[1]} columns")
filepath = dv.save_parquet(data=out_df, filename=OUTNAME, outpath=OUTPATH)

***

# End of notebook.

In [ ]:
NOTEBOOK_END = datetime.now()
_h, _rem = divmod(int((NOTEBOOK_END - NOTEBOOK_START).total_seconds()), 3600)
_m, _s = divmod(_rem, 60)

print(f"Started:  {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Finished: {NOTEBOOK_END.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Runtime:  {_h:02d}:{_m:02d}:{_s:02d} (hh:mm:ss)")